# CyberSentinel-LLM — Demo Notebook
**CMC 2025** | LLM-Driven Autonomous Cyber Threat Detection

Walkthrough: Data → Train → Evaluate → Inference + Forensic Report

## 1. Setup

In [ ]:
import sys, os; sys.path.insert(0, '..')
import torch, numpy as np
import matplotlib.pyplot as plt; %matplotlib inline
from config import THREAT_CLASSES, NUM_CLASSES
from dataset import build_dataloaders
from model import build_model, CyberSentinelLoss
from utils import set_seed, compute_metrics, plot_confusion_matrix, plot_roc_curves, print_metrics_table
set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 2. Load Data

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(use_synthetic=True, batch_size=32)
x, y = next(iter(train_loader))
print(f"Batch: x={x.shape}, y={y.shape}")
print(f"Classes: {THREAT_CLASSES}")

## 3. Model

In [ ]:
model = build_model(input_dim=x.shape[-1], device=device)

## 4. Train (5 demo epochs)

In [ ]:
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast

criterion = CyberSentinelLoss()
optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = GradScaler(enabled=(device != 'cpu'))
losses, accs = [], []

for epoch in range(1, 6):
    model.train(); ep_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        with autocast(enabled=(device != 'cpu')):
            out = model(xb); loss, _ = criterion(out['logits'], out['h_LLM'], yb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        ep_loss += loss.item()
    model.eval(); ps, ls, ss = [], [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device); out = model(xb)
            ps.append(out['logits'].argmax(-1).cpu().numpy())
            ls.append(yb.cpu().numpy())
            ss.append(torch.softmax(out['logits'],-1).cpu().numpy())
    preds, labels, scores = np.concatenate(ps), np.concatenate(ls), np.concatenate(ss)
    m = compute_metrics(labels, preds, scores, NUM_CLASSES)
    avg = ep_loss/len(train_loader); accs.append(m['accuracy']*100); losses.append(avg)
    print(f"Epoch {epoch}/5 | Loss={avg:.4f} | Val Acc={m['accuracy']*100:.2f}% | F1={m['f1_macro']*100:.2f}%")

## 5. Curves

In [ ]:
fig, (a1,a2) = plt.subplots(1,2,figsize=(10,4))
a1.plot(range(1,6), losses,'b-o',lw=2); a1.set_title('Train Loss'); a1.grid(alpha=0.3)
a2.plot(range(1,6), accs,'r-s',lw=2); a2.set_title('Val Accuracy (%)'); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('../results/curves.png', dpi=150); plt.show()

## 6. Test Evaluation

In [ ]:
model.eval(); ps,ls,ss=[],[],[]
with torch.no_grad():
    for xb,yb in test_loader:
        xb,yb=xb.to(device),yb.to(device); out=model(xb)
        ps.append(out['logits'].argmax(-1).cpu().numpy())
        ls.append(yb.cpu().numpy())
        ss.append(torch.softmax(out['logits'],-1).cpu().numpy())
preds,labels,scores=np.concatenate(ps),np.concatenate(ls),np.concatenate(ss)
metrics=compute_metrics(labels,preds,scores,NUM_CLASSES)
print_metrics_table(metrics,"Test Results")
os.makedirs('../results',exist_ok=True)
plot_confusion_matrix(labels,preds,THREAT_CLASSES,save_path='../results/cm.png')
plot_roc_curves(labels,scores,THREAT_CLASSES,save_path='../results/roc.png')
print("Saved: confusion matrix + ROC curves")

## 7. Inference + Forensic Report

In [ ]:
from inference import CyberSentinelInference
engine = CyberSentinelInference(checkpoint_path=None, input_dim=x.shape[-1], device=device)
apt_seq = np.random.default_rng(99).normal(2.5,0.4,(256,x.shape[-1])).astype('float32')
r = engine.predict(apt_seq)
print(f"Class: {r['class_name']} | Score: {r['detection_score']:.4f} | Anomaly: {'YES' if r['is_anomaly'] else 'NO'}")
print(f"Action: {r['action_name']}")
print(r['forensic_report'])

## Done!
- Full paper results: `python train.py --real_data --epochs 50`
- Evaluate: `python evaluate.py`